⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Migration of Dept_MAp\n",
    "**Source File:** Dept_MAp.sql\n",
    "**Conversion Date:** 2024-05-23\n",
    "**Target Platform:** Databricks Spark SQL / Delta Lake"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "dbutils.widgets.text(\"ODI_SESS_NO\", \"ba90851c-988d-4cb8-8c6d-b82450b55a67\")\n",
    "dbutils.widgets.text(\"v_lst_dt\", \"\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## ETL Parameters"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {1}: Get ETL last run date\n",
    "CREATE OR REPLACE TEMPORARY VIEW v_lst_dt AS\n",
    "SELECT \n",
    "    COALESCE(\n",
    "        (SELECT date_format(last_run_dt, 'dd-MM-yy') \n",
    "         FROM workspace.odi_trg.log_table1 \n",
    "         WHERE pkg_name = 'Dept_pkg'),\n",
    "        date_format(current_timestamp(), 'dd-MM-yy')\n",
    "    ) AS v_lst_dt;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "display(spark.sql(\"SELECT * FROM v_lst_dt\"))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {2}: Update log table last run date\n",
    "UPDATE workspace.odi_trg.log_table1 \n",
    "SET last_run_dt = current_timestamp() \n",
    "WHERE pkg_name = 'Dept_pkg';"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Staging Table: c_0filter_stg"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {30}: Drop staging table\n",
    "DROP TABLE IF EXISTS workspace.odi_trg.c_0filter_stg;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {40}: Create staging table\n",
    "CREATE TABLE workspace.odi_trg.c_0filter_stg\n",
    "(\n",
    "    DEPARTMENT_ID    BIGINT,\n",
    "    DEPARTMENT_NAME  STRING,\n",
    "    MANAGER_ID       BIGINT,\n",
    "    LOCATION_ID      BIGINT,\n",
    "    LAST_UPD_DT      TIMESTAMP\n",
    ") USING DELTA;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {50}: Insert into staging\n",
    "INSERT INTO workspace.odi_trg.c_0filter_stg\n",
    "(\n",
    "    DEPARTMENT_ID,\n",
    "    DEPARTMENT_NAME,\n",
    "    MANAGER_ID,\n",
    "    LOCATION_ID,\n",
    "    LAST_UPD_DT\n",
    ")\n",
    "SELECT \n",
    "    SRC_DEPARTMENTS_1.DEPARTMENT_ID,\n",
    "    SRC_DEPARTMENTS_1.DEPARTMENT_NAME,\n",
    "    SRC_DEPARTMENTS_1.MANAGER_ID,\n",
    "    SRC_DEPARTMENTS_1.LOCATION_ID,\n",
    "    SRC_DEPARTMENTS_1.LAST_UPD_DT\n",
    "FROM workspace.odi_src.src_departments AS SRC_DEPARTMENTS_1\n",
    "WHERE (1=1)\n",
    "AND (SRC_DEPARTMENTS_1.LAST_UPD_DT >= to_date((SELECT v_lst_dt FROM v_lst_dt), 'dd-MM-yy'));"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Flow Table: i_trg_departments_flow"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {70}: Drop flow table\n",
    "DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {80}: Create flow table\n",
    "CREATE TABLE workspace.odi_trg.i_trg_departments_flow\n",
    "(\n",
    "    DEPARTMENT_ID    BIGINT,\n",
    "    DEPARTMENT_NAME  STRING,\n",
    "    MANAGER_ID       BIGINT,\n",
    "    LOCATION_ID      BIGINT,\n",
    "    LAST_UPD_DT      TIMESTAMP,\n",
    "    IND_UPDATE       STRING,\n",
    "    ODI_ROW_ID       STRING\n",
    ") USING DELTA;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {90}: Insert into flow table (Detect Not Exists)\n",
    "INSERT INTO workspace.odi_trg.i_trg_departments_flow\n",
    "(\n",
    "    DEPARTMENT_ID,\n",
    "    DEPARTMENT_NAME,\n",
    "    MANAGER_ID,\n",
    "    LOCATION_ID,\n",
    "    LAST_UPD_DT,\n",
    "    IND_UPDATE,\n",
    "    ODI_ROW_ID\n",
    ")\n",
    "SELECT \n",
    "    S.DEPARTMENT_ID,\n",
    "    S.DEPARTMENT_NAME,\n",
    "    S.MANAGER_ID,\n",
    "    S.LOCATION_ID,\n",
    "    S.LAST_UPD_DT,\n",
    "    S.IND_UPDATE,\n",
    "    CAST(monotonically_increasing_id() AS STRING) AS ODI_ROW_ID\n",
    "FROM (\n",
    "    SELECT \n",
    "        FILTER_A.DEPARTMENT_ID,\n",
    "        FILTER_A.DEPARTMENT_NAME,\n",
    "        FILTER_A.MANAGER_ID,\n",
    "        FILTER_A.LOCATION_ID,\n",
    "        FILTER_A.LAST_UPD_DT,\n",
    "        'I' AS IND_UPDATE\n",
    "    FROM workspace.odi_trg.c_0filter_stg AS FILTER_A\n",
    "    WHERE (1=1)\n",
    ") AS S\n",
    "WHERE NOT EXISTS (\n",
    "    SELECT 1 FROM workspace.odi_trg.trg_departments AS T\n",
    "    WHERE T.DEPARTMENT_ID = S.DEPARTMENT_ID \n",
    "    AND (T.DEPARTMENT_NAME = S.DEPARTMENT_NAME OR (T.DEPARTMENT_NAME IS NULL AND S.DEPARTMENT_NAME IS NULL))\n",
    "    AND (T.MANAGER_ID = S.MANAGER_ID OR (T.MANAGER_ID IS NULL AND S.MANAGER_ID IS NULL))\n",
    "    AND (T.LOCATION_ID = S.LOCATION_ID OR (T.LOCATION_ID IS NULL AND S.LOCATION_ID IS NULL))\n",
    "    AND (T.LAST_UPD_DT = S.LAST_UPD_DT OR (T.LAST_UPD_DT IS NULL AND S.LAST_UPD_DT IS NULL))\n",
    ");"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {100} & {110}: Optimize and Gather Stats\n",
    "-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS\n",
    "SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;\n",
    "OPTIMIZE workspace.odi_trg.i_trg_departments_flow ZORDER BY (DEPARTMENT_ID);"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Error Handling and Validation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {120}: Create SNP_CHECK_TAB\n",
    "CREATE TABLE IF NOT EXISTS workspace.odi_trg.snp_check_tab\n",
    "(\n",
    "    CATALOG_NAME    STRING,\n",
    "    SCHEMA_NAME     STRING,\n",
    "    RESOURCE_NAME   STRING,\n",
    "    FULL_RES_NAME   STRING,\n",
    "    ERR_TYPE        STRING,\n",
    "    ERR_MESS        STRING,\n",
    "    CHECK_DATE      TIMESTAMP,\n",
    "    ORIGIN          STRING,\n",
    "    CONS_NAME       STRING,\n",
    "    CONS_TYPE       STRING,\n",
    "    ERR_COUNT       BIGINT\n",
    ") USING DELTA;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {130}: Cleanup SNP_CHECK_TAB\n",
    "DELETE FROM workspace.odi_trg.snp_check_tab\n",
    "WHERE SCHEMA_NAME = 'ODI_TRG'\n",
    "AND ORIGIN = '(7)test.Dept_MAp'\n",
    "AND ERR_TYPE = 'F';"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {140}: Create E$_TRG_DEPARTMENTS\n",
    "CREATE TABLE IF NOT EXISTS workspace.odi_trg.e_trg_departments\n",
    "(\n",
    "    ODI_ROW_ID      STRING,\n",
    "    ODI_ERR_TYPE    STRING,\n",
    "    ODI_ERR_MESS    STRING,\n",
    "    ODI_CHECK_DATE  TIMESTAMP,\n",
    "    DEPARTMENT_ID   BIGINT,\n",
    "    DEPARTMENT_NAME STRING,\n",
    "    MANAGER_ID      BIGINT,\n",
    "    LOCATION_ID     BIGINT,\n",
    "    LAST_UPD_DT     TIMESTAMP,\n",
    "    ODI_ORIGIN      STRING,\n",
    "    ODI_CONS_NAME   STRING,\n",
    "    ODI_CONS_TYPE   STRING,\n",
    "    ODI_PK          STRING,\n",
    "    ODI_SESS_NO     STRING\n",
    ") USING DELTA;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {150}: Cleanup E$ table\n",
    "DELETE FROM workspace.odi_trg.e_trg_departments\n",
    "WHERE (ODI_ERR_TYPE = 'S' AND 1 = 0)\n",
    "OR (ODI_ERR_TYPE = 'F' AND ODI_ORIGIN = '(7)test.Dept_MAp');"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {170}: PK Violation Detection\n",
    "INSERT INTO workspace.odi_trg.e_trg_departments\n",
    "(\n",
    "    ODI_PK,\n",
    "    ODI_SESS_NO,\n",
    "    ODI_ROW_ID,\n",
    "    ODI_ERR_TYPE,\n",
    "    ODI_ERR_MESS,\n",
    "    ODI_ORIGIN,\n",
    "    ODI_CHECK_DATE,\n",
    "    ODI_CONS_NAME,\n",
    "    ODI_CONS_TYPE,\n",
    "    DEPARTMENT_ID,\n",
    "    DEPARTMENT_NAME,\n",
    "    MANAGER_ID,\n",
    "    LOCATION_ID,\n",
    "    LAST_UPD_DT\n",
    ")\n",
    "SELECT \n",
    "    uuid(),\n",
    "    '${ODI_SESS_NO}',\n",
    "    TRG_DEPARTMENTS.ODI_ROW_ID,\n",
    "    'F',\n",
    "    'ODI-15064: The primary key PK_department_id is not unique.',\n",
    "    '(7)test.Dept_MAp',\n",
    "    current_timestamp(),\n",
    "    'PK_department_id',\n",
    "    'PK',\n",
    "    TRG_DEPARTMENTS.DEPARTMENT_ID,\n",
    "    TRG_DEPARTMENTS.DEPARTMENT_NAME,\n",
    "    TRG_DEPARTMENTS.MANAGER_ID,\n",
    "    TRG_DEPARTMENTS.LOCATION_ID,\n",
    "    TRG_DEPARTMENTS.LAST_UPD_DT\n",
    "FROM workspace.odi_trg.i_trg_departments_flow AS TRG_DEPARTMENTS\n",
    "WHERE EXISTS (\n",
    "    SELECT SUB.DEPARTMENT_ID\n",
    "    FROM workspace.odi_trg.i_trg_departments_flow AS SUB\n",
    "    WHERE SUB.DEPARTMENT_ID = TRG_DEPARTMENTS.DEPARTMENT_ID\n",
    "    GROUP BY SUB.DEPARTMENT_ID\n",
    "    HAVING count(1) > 1\n",
    ");"
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {180}: Not Null Check (DEPARTMENT_NAME)\n",
    "INSERT INTO workspace.odi_trg.e_trg_departments\n",
    "(\n",
    "    ODI_PK,\n",
    "    ODI_SESS_NO,\n",
    "    ODI_ROW_ID,\n",
    "    ODI_ERR_TYPE,\n",
    "    ODI_ERR_MESS,\n",
    "    ODI_CHECK_DATE,\n",
    "    ODI_ORIGIN,\n",
    "    ODI_CONS_NAME,\n",
    "    ODI_CONS_TYPE,\n",
    "    DEPARTMENT_ID,\n",
    "    DEPARTMENT_NAME,\n",
    "    MANAGER_ID,\n",
    "    LOCATION_ID,\n",
    "    LAST_UPD_DT\n",
    ")\n",
    "SELECT\n",
    "    uuid(),\n",
    "    '${ODI_SESS_NO}',\n",
    "    ODI_ROW_ID,\n",
    "    'F',\n",
    "    'ODI-15066: The column DEPARTMENT_NAME cannot be null.',\n",
    "    current_timestamp(),\n",
    "    '(7)test.Dept_MAp',\n",
    "    'DEPARTMENT_NAME',\n",
    "    'NN',\n",
    "    DEPARTMENT_ID,\n",
    "    DEPARTMENT_NAME,\n",
    "    MANAGER_ID,\n",
    "    LOCATION_ID,\n",
    "    LAST_UPD_DT\n",
    "FROM workspace.odi_trg.i_trg_departments_flow\n",
    "WHERE DEPARTMENT_NAME IS NULL;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {200}: Remove Erroneous Records from Flow\n",
    "MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T\n",
    "USING (\n",
    "    SELECT ODI_ROW_ID \n",
    "    FROM workspace.odi_trg.e_trg_departments\n",
    "    WHERE ODI_SESS_NO = '${ODI_SESS_NO}'\n",
    ") AS S\n",
    "ON T.ODI_ROW_ID = S.ODI_ROW_ID\n",
    "WHEN MATCHED THEN DELETE;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {210}: Insert Summary into snp_check_tab\n",
    "INSERT INTO workspace.odi_trg.snp_check_tab\n",
    "(\n",
    "    SCHEMA_NAME,\n",
    "    RESOURCE_NAME,\n",
    "    FULL_RES_NAME,\n",
    "    ERR_TYPE,\n",
    "    ERR_MESS,\n",
    "    CHECK_DATE,\n",
    "    ORIGIN,\n",
    "    CONS_NAME,\n",
    "    CONS_TYPE,\n",
    "    ERR_COUNT\n",
    ")\n",
    "SELECT\n",
    "    'ODI_TRG',\n",
    "    'TRG_DEPARTMENTS',\n",
    "    'workspace.odi_trg.trg_departments',\n",
    "    E.ODI_ERR_TYPE,\n",
    "    E.ODI_ERR_MESS,\n",
    "    E.ODI_CHECK_DATE,\n",
    "    E.ODI_ORIGIN,\n",
    "    E.ODI_CONS_NAME,\n",
    "    E.ODI_CONS_TYPE,\n",
    "    count(1)\n",
    "FROM workspace.odi_trg.e_trg_departments AS E\n",
    "WHERE E.ODI_ERR_TYPE = 'F'\n",
    "AND E.ODI_ORIGIN = '(7)test.Dept_MAp'\n",
    "GROUP BY\n",
    "    E.ODI_ERR_TYPE,\n",
    "    E.ODI_ERR_MESS,\n",
    "    E.ODI_CHECK_DATE,\n",
    "    E.ODI_ORIGIN,\n",
    "    E.ODI_CONS_NAME,\n",
    "    E.ODI_CONS_TYPE;"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Final Integration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {220}: Mark records for Update\n",
    "MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T\n",
    "USING (\n",
    "    SELECT DEPARTMENT_ID FROM workspace.odi_trg.trg_departments\n",
    ") AS S\n",
    "ON T.DEPARTMENT_ID = S.DEPARTMENT_ID\n",
    "WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {260} + {270}: MERGE into Target Table\n",
    "MERGE INTO workspace.odi_trg.trg_departments AS T\n",
    "USING workspace.odi_trg.i_trg_departments_flow AS S\n",
    "ON T.DEPARTMENT_ID = S.DEPARTMENT_ID\n",
    "WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET\n",
    "    T.DEPARTMENT_NAME = S.DEPARTMENT_NAME,\n",
    "    T.MANAGER_ID      = S.MANAGER_ID,\n",
    "    T.LOCATION_ID     = S.LOCATION_ID,\n",
    "    T.LAST_UPD_DT     = S.LAST_UPD_DT\n",
    "WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT\n",
    "(\n",
    "    DEPARTMENT_ID,\n",
    "    DEPARTMENT_NAME,\n",
    "    MANAGER_ID,\n",
    "    LOCATION_ID,\n",
    "    LAST_UPD_DT\n",
    ")\n",
    "VALUES\n",
    "(\n",
    "    S.DEPARTMENT_ID,\n",
    "    S.DEPARTMENT_NAME,\n",
    "    S.MANAGER_ID,\n",
    "    S.LOCATION_ID,\n",
    "    S.LAST_UPD_DT\n",
    ");"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cleanup"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {330}: Drop staging table\n",
    "DROP TABLE IF EXISTS workspace.odi_trg.c_0filter_stg;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {350}: Drop flow table\n",
    "DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}